# VC Project — Free-Data Deal Sourcing Pipeline

**Author:** Eesha Jagdhane · © 2026 · All rights reserved.  
**Original repository:** https://github.com/eeshajagdhane/vc-deal-sourcing

If this notebook is not running from that repository, it is a copy of the original.

**What this notebook does, in one sentence:** it finds real startups a venture-capital
investor might want to look at, using only **free, public data** — no paid subscriptions
(no PitchBook, no Harmonic, no Crunchbase Pro).

This is a rebuild of an earlier project that used paid data. Here we prove the same idea
works with free sources, and we explain every step in plain English so a non-technical
reader can follow along.

### How the notebook is organized
The notebook is built up **one milestone at a time**. Each milestone is a section below.
You can run the whole notebook top-to-bottom, and each section explains what it's doing
and why before it does it.

| Milestone | What it does | Status |
|---|---|---|
| **1. Discovery** | Find real companies from SEC EDGAR + Y Combinator | ✅ built |
| 2. Clean & de-duplicate | Merge duplicate records into one clean company | ⏳ next |
| 3. Enrichment | Add GitHub / Hacker News / job-posting signals | ⏳ later |
| 4+. Scoring, summaries, serving | Rank & explain the best-fit companies | 🗺️ roadmap |

> **How to run:** click *Kernel → Restart & Run All*, or run each cell top to bottom with
> Shift+Enter. The Discovery section makes live calls to public APIs, so it needs internet
> and takes a couple of minutes the first time. Results are cached so re-runs are instant.

## Section 0 — Setup & Configuration

First we import the Python libraries we need and set up a few **configuration values** you
can change. Think of `CONFIG` as the "control panel" for the whole notebook: your
investment thesis, which company stages/regions you care about, and how much data to pull.

You don't need to understand the library imports — just know that:
- `requests` lets us download data from the internet (the free APIs).
- `pandas` is a spreadsheet-like tool for holding tables of companies.
- The rest help us parse dates, web addresses, and XML files.

In [33]:
# Copyright (c) 2026 Eesha Jagdhane. All rights reserved.
# Original: https://github.com/eeshajagdhane/vc-deal-sourcing
# --- Standard Python tools ---
import os
import json
import time
import warnings
import xml.etree.ElementTree as ET  # for reading SEC's XML filing format
from collections import Counter

# Hide one harmless pandas "FutureWarning" so the notebook output stays clean.
warnings.filterwarnings("ignore", category=FutureWarning)

# --- Third-party tools (installed via requirements.txt) ---
import requests  # download data from the web
import pandas as pd  # tables of data

# Make results reproducible / tidy
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

print("Libraries loaded.")
print("Original repository: https://github.com/eeshajagdhane/vc-deal-sourcing")
print("Copyright (c) 2026 Eesha Jagdhane. All rights reserved.")

Libraries loaded.


In [34]:
# CONFIG — the "control panel". Edit these values to change behavior.

CONFIG = {
    # Your investment thesis in plain words. (Used later, in the scoring milestone.)
    "thesis": "AI infrastructure and developer tools; Pre-Seed to Series A; US-based",
    # Which funding stages and regions you care about (used later, in scoring).
    "target_stages": ["Pre-Seed", "Seed", "Early"],
    "target_geos": ["US"],
    # ---- SEC EDGAR settings (source #1: companies raising money) ----
    "edgar": {
        # How far back to look for funding filings, in days.
        "lookback_days": 90,
        # SEC asks automated tools to identify themselves with a name + email.
        # Please keep your own email here.
        "user_agent": "VC Project Research eeshajagdhane@ucsd.edu",
        # To keep the notebook fast, we only *scan* up to this many filings.
        # Raise it to cast a wider net (slower). Each filing is one small download.
        "max_filings_to_scan": 200,
        # SEC's Form D data labels each filer with an "industry group". We only keep
        # groups that represent actual operating startups (and drop investment funds,
        # real-estate deals, etc., which also file Form D but aren't companies to invest in).
        # Widen this list to see more sectors.
        "keep_industry_groups": [
            "Technology",
            "Other Technology",
            "Computers",
            "Telecommunications",
            "Biotechnology",
            "Pharmaceuticals",
            "Other Health Care",
            "Health Insurance",
            "Business Services",
            "Manufacturing",
            "Other",
        ],
    },
    # ---- Y Combinator directory settings (source #2: known startups) ----
    "yc": {
        # A free, public, no-login JSON list of every company YC has funded.
        "url": "https://yc-oss.github.io/api/companies/all.json",
        # Only keep YC companies still marked "Active" (skip acquired/dead ones).
        "only_active": True,
        # To keep things manageable we keep the most recent N YC companies.
        "max_companies": 400,
    },
}

# Where we save things
RAW_DIR = "data/raw"  # cached downloads (so we don't re-hit the APIs every run)
OUTPUT_DIR = "output"  # final results
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# A polite header sent with every SEC request (identifies us, as SEC requires).
EDGAR_HEADERS = {"User-Agent": CONFIG["edgar"]["user_agent"]}

print("Config ready. Thesis:", CONFIG["thesis"])

Config ready. Thesis: AI infrastructure and developer tools; Pre-Seed to Series A; US-based


## Section 0.1 — The "canonical schema" (one standard shape for every company)

Different data sources describe companies differently. SEC filings give us funding
amounts; YC gives us websites and descriptions. To keep everything tidy, we convert every
company — no matter where it came from — into **one standard set of fields**. This is
called the *canonical schema*.

A field can be blank if a particular source didn't provide it (e.g., SEC doesn't give a
website). That's fine — later milestones fill in the gaps.

In [35]:
# The standard fields every company record will have.
CANONICAL_FIELDS = [
    "company_id",  # a stable ID we create for each company
    "name",
    "website_domain",  # e.g. "stripe.com"
    "description",
    "industry_primary",
    "verticals",  # sector tags / keywords
    "year_founded",
    "hq_city",
    "hq_state",
    "hq_country",
    "funding_total_usd",
    "last_round_type",  # inferred stage: Pre-Seed / Seed / Early / Later
    "last_round_size_usd",  # the offering amount from the SEC filing
    "last_round_date",
    "filing_count",  # how many SEC Form D filings we saw for this company
    "source",  # which source this row came from: "edgar" or "yc"
]

print("A company record has these fields:")
for f in CANONICAL_FIELDS:
    print("  -", f)

A company record has these fields:
  - company_id
  - name
  - website_domain
  - description
  - industry_primary
  - verticals
  - year_founded
  - hq_city
  - hq_state
  - hq_country
  - funding_total_usd
  - last_round_type
  - last_round_size_usd
  - last_round_date
  - filing_count
  - source


---
# Milestone 1 — Discovery: finding real companies

**Goal:** build a first list of real companies to consider, pulled from two free sources.

### Source #1: SEC EDGAR "Form D" filings — *companies actively raising money*
When a US startup raises private investment above a certain size, it is **legally required**
to file a short form (called **Form D**) with the SEC (the government's financial regulator).
These filings are public. Each one tells us: the company's name, where it's based, roughly
when it was founded, its industry, and **how much money it's raising**. That last part is
gold — it's a real, current signal that a company is fundraising *right now*.

A catch: lots of things file Form D that aren't startups — investment funds, real-estate
deals, etc. We filter those out using the "industry group" the SEC records on each filing.

### Source #2: Y Combinator directory — *a known list of vetted startups*
Y Combinator is a famous startup accelerator. It publishes a public directory of every
company it has funded, including each one's website, a short description, and its sector.
We use a free, public copy of that directory (a plain JSON file hosted on GitHub — no login
or key needed).

We'll pull from both, convert each into our standard company shape, and combine them into a
single table: `output/companies_raw.csv`.

### 1a. Load SEC EDGAR Form D filings

The code below does three things:
1. **Enumerate** recent Form D filings (just the list of filings in our date window).
2. For each filing, **download and read** the small XML form to pull out the useful fields.
3. **Filter** down to real operating startups (dropping funds/real-estate) and infer a
   rough funding *stage* from the dollar amount.

We cache the raw result to `data/raw/edgar_formd.json` so that re-running the notebook
doesn't re-download everything.

In [36]:
from datetime import date, timedelta

EDGAR_SEARCH_URL = "https://efts.sec.gov/LATEST/search-index"


def edgar_enumerate_filings(start_date, end_date, max_filings, headers):
    """Return a list of recent Form D filings (metadata only) between two dates.

    We ask SEC's full-text search for all form-type-D filings in the window. It returns
    results in pages of 10, so we page through until we have enough.
    """
    results = []
    offset = 0
    while len(results) < max_filings:
        params = {
            "q": "",  # empty query = "all filings" (we filter by form + date)
            "forms": "D",
            "dateRange": "custom",
            "startdt": start_date,
            "enddt": end_date,
            "from": offset,
        }
        resp = requests.get(
            EDGAR_SEARCH_URL, params=params, headers=headers, timeout=30
        )
        resp.raise_for_status()
        hits = resp.json().get("hits", {}).get("hits", [])
        if not hits:
            break  # no more pages
        results.extend(hits)
        offset += len(hits)
        time.sleep(0.2)  # be polite: pause between requests
    return results[:max_filings]


def edgar_parse_filing(hit, headers):
    """Download one Form D filing's XML and pull out the fields we care about."""
    source = hit["_source"]
    accession, doc_name = hit["_id"].split(
        ":"
    )  # e.g. "0001234567-24-000001:primary_doc.xml"
    cik = source["ciks"][0].lstrip("0")  # the company's SEC id
    url = f"https://www.sec.gov/Archives/edgar/data/{cik}/{accession.replace('-', '')}/{doc_name}"
    xml_text = requests.get(url, headers=headers, timeout=30).text
    root = ET.fromstring(xml_text)

    def get(path):
        el = root.find(path)
        return el.text if el is not None else None

    issuer = "primaryIssuer/"
    return {
        "name": get(issuer + "entityName"),
        "hq_city": get(issuer + "issuerAddress/city"),
        "hq_state": get(issuer + "issuerAddress/stateOrCountry"),
        "year_founded": get(issuer + "yearOfInc/value"),
        "entity_type": get(issuer + "entityType"),
        "industry_group": get("offeringData/industryGroup/industryGroupType"),
        # "totalOfferingAmount" = how much they're trying to raise (can be "Indefinite")
        "offering_amount": get("offeringData/offeringSalesAmounts/totalOfferingAmount"),
        "amount_sold": get("offeringData/offeringSalesAmounts/totalAmountSold"),
        "file_date": source.get("file_date"),
        "cik": cik,
    }


print("EDGAR helper functions defined.")

EDGAR helper functions defined.


Now we infer a rough **funding stage** from the offering amount. SEC filings don't say
"Seed" or "Series A" — they just give a dollar figure. So we bucket the amount into a stage
using typical US venture ranges. These are rough guesses, clearly, but good enough to sort
companies by how far along they are.

In [37]:
# Rough US venture ranges. [low, high] in dollars; high=None means "no upper limit".
STAGE_BUCKETS = {
    "Pre-Seed": (0, 500_000),
    "Seed": (500_000, 3_000_000),
    "Early": (3_000_000, 15_000_000),  # ~ Series A
    "Later": (15_000_000, None),  # Series B and beyond
}


def infer_stage(offering_amount):
    """Turn a dollar amount (or the word 'Indefinite') into a stage label."""
    # SEC sometimes reports "Indefinite" instead of a number -> we can't tell, call it Seed.
    try:
        amount = float(offering_amount)
    except (TypeError, ValueError):
        return "Seed"
    for stage, (low, high) in STAGE_BUCKETS.items():
        if amount >= low and (high is None or amount < high):
            return stage
    return "Seed"


def to_number(x):
    """Safely turn a string like '16999997' into a number, or None if it isn't one."""
    try:
        return float(x)
    except (TypeError, ValueError):
        return None


print("Stage-inference ready. Example: $17M ->", infer_stage("16999997"))

Stage-inference ready. Example: $17M -> Later


In [38]:
def load_edgar_companies(config, headers, use_cache=True):
    """Full EDGAR pipeline: enumerate -> download -> filter -> standardize."""
    cache_path = os.path.join(RAW_DIR, "edgar_formd.json")

    # Use cached raw filings if we already downloaded them (keeps re-runs fast).
    if use_cache and os.path.exists(cache_path):
        with open(cache_path) as f:
            parsed = json.load(f)
        print(f"Loaded {len(parsed)} cached EDGAR filings from {cache_path}")
    else:
        end = date.today()
        start = end - timedelta(days=config["edgar"]["lookback_days"])
        print(f"Enumerating Form D filings from {start} to {end} ...")
        hits = edgar_enumerate_filings(
            str(start), str(end), config["edgar"]["max_filings_to_scan"], headers
        )
        print(f"Found {len(hits)} filings; downloading and reading each one...")
        parsed = []
        for i, hit in enumerate(hits):
            try:
                parsed.append(edgar_parse_filing(hit, headers))
            except Exception as e:
                print("  (skipped one filing due to error:", e, ")")
            time.sleep(0.12)  # be polite to SEC's servers
            if (i + 1) % 50 == 0:
                print(f"  ...read {i + 1}/{len(hits)}")
        with open(cache_path, "w") as f:
            json.dump(parsed, f)
        print(f"Saved raw filings to {cache_path}")

    # Show what kinds of filers we saw (mostly funds + real estate; startups are the minority).
    print("\nIndustry groups seen:", dict(Counter(p["industry_group"] for p in parsed)))

    # Keep only operating-startup industry groups, and drop obvious non-startups.
    keep = set(config["edgar"]["keep_industry_groups"])
    rows = []
    for p in parsed:
        if p.get("industry_group") not in keep:
            continue
        if (p.get("entity_type") or "").lower().startswith("limited partnership"):
            continue  # LPs are almost always funds, not startups
        amount = to_number(p.get("offering_amount"))
        rows.append(
            {
                "name": p["name"],
                "website_domain": None,  # EDGAR doesn't give websites
                "description": None,  # ...or descriptions
                "industry_primary": p.get("industry_group"),
                "verticals": None,
                "year_founded": p.get("year_founded"),
                "hq_city": p.get("hq_city"),
                "hq_state": p.get("hq_state"),
                "hq_country": "US",  # Form D issuers are US-registered
                "funding_total_usd": amount,
                "last_round_type": infer_stage(p.get("offering_amount")),
                "last_round_size_usd": amount,
                "last_round_date": p.get("file_date"),
                "filing_count": 1,
                "source": "edgar",
            }
        )
    df = pd.DataFrame(rows, columns=[c for c in CANONICAL_FIELDS if c != "company_id"])
    print(f"\nKept {len(df)} operating-startup filings after filtering.")
    return df


edgar_df = load_edgar_companies(CONFIG, EDGAR_HEADERS)
edgar_df.head(10)

Loaded 200 cached EDGAR filings from data/raw/edgar_formd.json

Industry groups seen: {'Pooled Investment Fund': 129, 'Other': 12, 'Other Technology': 12, 'Commercial': 4, 'Residential': 4, 'Other Travel': 1, 'Other Real Estate': 4, 'Business Services': 2, 'Agriculture': 2, 'REITS and Finance': 4, 'Manufacturing': 5, 'Retailing': 2, 'Investing': 4, 'Other Energy': 3, 'Computers': 2, 'Restaurants': 1, 'Other Banking and Financial Services': 3, 'Biotechnology': 5, 'Pharmaceuticals': 1}

Kept 38 operating-startup filings after filtering.


,name,website_domain,description,industry_primary,verticals,year_founded,hq_city,hq_state,hq_country,funding_total_usd,last_round_type,last_round_size_usd,last_round_date,filing_count,source
0,"ReLiftMD Ventures, LLC",None,None,Other,None,2024,MILLERSVILLE,MD,US,NaN,Seed,NaN,2026-07-28,1,edgar
1,Wayy Inc.,None,None,Other Technology,None,2023,MIAMI,FL,US,2000000.0,Seed,2000000.0,2026-07-28,1,edgar
2,"Kiken Technoloigies, Inc",None,None,Other Technology,None,2026,MERRILLVILLE,IN,US,131000.0,Pre-Seed,131000.0,2026-07-28,1,edgar
3,i2 Health Advisors Inc.,None,None,Other Technology,None,2026,SEATTLE,WA,US,3500000.0,Early,3500000.0,2026-07-28,1,edgar
4,1541 Productions Ltd Liability Co,None,None,Other,None,2026,NEW YORK,NY,US,3000000.0,Early,3000000.0,2026-07-28,1,edgar
5,Auxesys Inc.,None,None,Business Services,None,None,OAKBANK,A2,US,300000.0,Pre-Seed,300000.0,2026-07-28,1,edgar
6,Bootheo Inc.,None,None,Other Technology,None,2026,ORLANDO,FL,US,150000.0,Pre-Seed,150000.0,2026-07-28,1,edgar
7,"Enhance Health, Inc.",None,None,Other,None,2023,FAYETTEVILLE,AR,US,145000.0,Pre-Seed,145000.0,2026-07-28,1,edgar
8,"VIN-59 Enhance Health, LLC",None,None,Other,None,2026,FAYETTEVILLE,AR,US,45000.0,Pre-Seed,45000.0,2026-07-28,1,edgar
9,"Oboro Labs, Inc.",None,None,Manufacturing,None,2024,DELRAY BEACH,FL,US,3500000.0,Early,3500000.0,2026-07-28,1,edgar


### 1b. Load the Y Combinator directory

This is simpler: we download one public JSON file listing every YC company, keep the active
ones, and standardize them into our shape. YC gives us the **website** and a **description**
(which EDGAR doesn't), so the two sources complement each other nicely.

In [39]:
import tldextract  # helps turn a full web address into a clean domain like "stripe.com"


def domain_from_url(url):
    """Turn 'https://www.stripe.com/about' into 'stripe.com'. Returns None if invalid."""
    if not isinstance(url, str) or not url.strip():
        return None
    ext = tldextract.extract(url)
    if not ext.domain or not ext.suffix:
        return None
    return f"{ext.domain}.{ext.suffix}".lower()


def load_yc_companies(config, headers, use_cache=True):
    """Download the public YC company directory and standardize it."""
    cache_path = os.path.join(RAW_DIR, "yc_companies.json")

    if use_cache and os.path.exists(cache_path):
        with open(cache_path) as f:
            data = json.load(f)
        print(f"Loaded {len(data)} cached YC companies from {cache_path}")
    else:
        print("Downloading YC company directory...")
        data = requests.get(config["yc"]["url"], headers=headers, timeout=60).json()
        with open(cache_path, "w") as f:
            json.dump(data, f)
        print(f"Downloaded {len(data)} YC companies; cached to {cache_path}")

    if config["yc"]["only_active"]:
        data = [c for c in data if c.get("status") == "Active"]

    # Keep the most recently launched companies (newest are most relevant to source now).
    data = sorted(data, key=lambda c: c.get("launched_at") or 0, reverse=True)
    data = data[: config["yc"]["max_companies"]]

    rows = []
    for c in data:
        location = c.get("all_locations") or ""
        parts = [p.strip() for p in location.split(",")]
        rows.append(
            {
                "name": c.get("name"),
                "website_domain": domain_from_url(c.get("website")),
                "description": c.get("long_description") or c.get("one_liner"),
                "industry_primary": c.get("industry"),
                "verticals": ", ".join(c.get("tags", [])) if c.get("tags") else None,
                "year_founded": None,  # YC lists a batch, not a founding year
                "hq_city": parts[0] if parts and parts[0] else None,
                "hq_state": parts[1] if len(parts) > 1 else None,
                "hq_country": parts[-1] if parts and parts[-1] else None,
                "funding_total_usd": None,  # YC directory doesn't list funding
                "last_round_type": None,
                "last_round_size_usd": None,
                "last_round_date": None,
                "filing_count": 0,
                "source": "yc",
            }
        )
    df = pd.DataFrame(rows, columns=[c for c in CANONICAL_FIELDS if c != "company_id"])
    print(f"Kept {len(df)} active YC companies.")
    return df


yc_df = load_yc_companies(CONFIG, EDGAR_HEADERS)
yc_df.head(10)

Loaded 6087 cached YC companies from data/raw/yc_companies.json
Kept 400 active YC companies.


,name,website_domain,description,industry_primary,verticals,year_founded,hq_city,hq_state,hq_country,funding_total_usd,last_round_type,last_round_size_usd,last_round_date,filing_count,source
0,Gutgutgoose,gutgutgoose.com,Most data companies pay for their data. Ours p...,Healthcare,"Biotech, Personalization, Health & Wellness",None,San Francisco,CA,USA,None,None,None,None,0,yc
1,Wondering,wondering.app,"Wondering turns anything you want to learn, fr...",Consumer,"AI-Enhanced Learning, Artificial Intelligence,...",None,San Francisco,CA,USA,None,None,None,None,0,yc
2,Synapse Semiconductor,synapsesemi.org,We want to push machines to where humans cant ...,Industrials,"Artificial Intelligence, Edge Computing Semico...",None,Durham,NC,USA,None,None,None,None,0,yc
3,Graphify Labs,graphify.com,Graphify builds queryable knowledge graphs fro...,B2B,"Developer Tools, Reinforcement Learning, Open ...",None,London,England,United Kingdom,None,None,None,None,0,yc
4,PokerClubHub,pokerclubhub.com,TL;DR: PokerClubHub is the first licensed poke...,Consumer,"SaaS, Crypto / Web3, Gaming, eSports",None,Jaco,Puntarenas Province,Costa Rica,None,None,None,None,0,yc
5,Traceforce,traceforce.ai,AI apps such as ChatGPT and Claude have become...,B2B,"Artificial Intelligence, SaaS, Security, Cyber...",None,San Francisco,CA,USA,None,None,None,None,0,yc
6,Lumeria,lumeria.skin,"Your skin is yours, but understanding it has n...",Healthcare,"Consumer Health Services, Health Tech, Health ...",None,San Francisco,CA,USA,None,None,None,None,0,yc
7,Atomarine,atomarine.co,We are building nuclear power delivery ships t...,Industrials,"Hard Tech, Energy",None,Boston,MA,USA,None,None,None,None,0,yc
8,Agency Tool Company,agencytool.com,Agency Tool Company ships software to robots i...,Industrials,"Developer Tools, Robotics, Automation, Infrast...",None,Denver,CO,USA,None,None,None,None,0,yc
9,RonanRx Inc.,ronanrx.com,RonanRX helps you personalize GLP-1s like Ozem...,Healthcare,"Health Tech, Telehealth, Therapeutics",None,San Francisco,CA,USA,None,None,None,None,0,yc


### 1c. Combine both sources into one raw table

Finally we stack the two tables on top of each other, give every company a stable ID, and
save the result to `output/companies_raw.csv`. This is the raw universe of companies —
duplicates and all. Cleaning up duplicates is the **next** milestone; for now we just want
the full list from both sources.

In [40]:
import hashlib


def make_company_id(row):
    """Create a stable ID for a company. Prefer its website; fall back to name + state."""
    basis = row["website_domain"] or f"{row['name']}|{row.get('hq_state') or ''}"
    return "c_" + hashlib.md5(str(basis).lower().encode()).hexdigest()[:12]


# Stack EDGAR + YC rows into one table.
companies_raw = pd.concat([edgar_df, yc_df], ignore_index=True)

# Drop rows with no company name (can't do anything with those).
companies_raw = companies_raw[companies_raw["name"].notna()].copy()

# Give each row a stable ID and put that column first.
companies_raw["company_id"] = companies_raw.apply(make_company_id, axis=1)
companies_raw = companies_raw[CANONICAL_FIELDS]

# Save it.
out_path = os.path.join(OUTPUT_DIR, "companies_raw.csv")
companies_raw.to_csv(out_path, index=False)

print(f"Saved {len(companies_raw)} companies to {out_path}")
print("\nBreakdown by source:")
print(companies_raw["source"].value_counts())
companies_raw.head(12)

Saved 438 companies to output/companies_raw.csv

Breakdown by source:
source
yc       400
edgar     38
Name: count, dtype: int64


,company_id,name,website_domain,description,industry_primary,verticals,year_founded,hq_city,hq_state,hq_country,funding_total_usd,last_round_type,last_round_size_usd,last_round_date,filing_count,source
0,c_aee526a8c850,"ReLiftMD Ventures, LLC",None,None,Other,None,2024,MILLERSVILLE,MD,US,NaN,Seed,NaN,2026-07-28,1,edgar
1,c_2ad0190d81b2,Wayy Inc.,None,None,Other Technology,None,2023,MIAMI,FL,US,2000000.0,Seed,2000000.0,2026-07-28,1,edgar
2,c_e03e94bb9188,"Kiken Technoloigies, Inc",None,None,Other Technology,None,2026,MERRILLVILLE,IN,US,131000.0,Pre-Seed,131000.0,2026-07-28,1,edgar
3,c_c3e03d3f0a42,i2 Health Advisors Inc.,None,None,Other Technology,None,2026,SEATTLE,WA,US,3500000.0,Early,3500000.0,2026-07-28,1,edgar
4,c_aedb312cece0,1541 Productions Ltd Liability Co,None,None,Other,None,2026,NEW YORK,NY,US,3000000.0,Early,3000000.0,2026-07-28,1,edgar
5,c_c862bcaf5389,Auxesys Inc.,None,None,Business Services,None,None,OAKBANK,A2,US,300000.0,Pre-Seed,300000.0,2026-07-28,1,edgar
6,c_c78ef6d4011f,Bootheo Inc.,None,None,Other Technology,None,2026,ORLANDO,FL,US,150000.0,Pre-Seed,150000.0,2026-07-28,1,edgar
7,c_b9dae5db247a,"Enhance Health, Inc.",None,None,Other,None,2023,FAYETTEVILLE,AR,US,145000.0,Pre-Seed,145000.0,2026-07-28,1,edgar
8,c_3e855deaa569,"VIN-59 Enhance Health, LLC",None,None,Other,None,2026,FAYETTEVILLE,AR,US,45000.0,Pre-Seed,45000.0,2026-07-28,1,edgar
9,c_e20216f248d3,"Oboro Labs, Inc.",None,None,Manufacturing,None,2024,DELRAY BEACH,FL,US,3500000.0,Early,3500000.0,2026-07-28,1,edgar


### 1d. Quick sanity check

Let's look at a few example companies from each source to confirm the data looks real and
sensible.

In [41]:
print("=== A few companies raising money right now (from SEC EDGAR) ===")
cols = [
    "name",
    "industry_primary",
    "hq_state",
    "last_round_type",
    "last_round_size_usd",
    "last_round_date",
]
display(companies_raw[companies_raw["source"] == "edgar"][cols].head(8))

print("\n=== A few YC startups (with websites + descriptions) ===")
cols2 = ["name", "website_domain", "industry_primary", "verticals"]
display(companies_raw[companies_raw["source"] == "yc"][cols2].head(8))

=== A few companies raising money right now (from SEC EDGAR) ===


,name,industry_primary,hq_state,last_round_type,last_round_size_usd,last_round_date
0,"ReLiftMD Ventures, LLC",Other,MD,Seed,NaN,2026-07-28
1,Wayy Inc.,Other Technology,FL,Seed,2000000.0,2026-07-28
2,"Kiken Technoloigies, Inc",Other Technology,IN,Pre-Seed,131000.0,2026-07-28
3,i2 Health Advisors Inc.,Other Technology,WA,Early,3500000.0,2026-07-28
4,1541 Productions Ltd Liability Co,Other,NY,Early,3000000.0,2026-07-28
5,Auxesys Inc.,Business Services,A2,Pre-Seed,300000.0,2026-07-28
6,Bootheo Inc.,Other Technology,FL,Pre-Seed,150000.0,2026-07-28
7,"Enhance Health, Inc.",Other,AR,Pre-Seed,145000.0,2026-07-28



=== A few YC startups (with websites + descriptions) ===


,name,website_domain,industry_primary,verticals
38,Gutgutgoose,gutgutgoose.com,Healthcare,"Biotech, Personalization, Health & Wellness"
39,Wondering,wondering.app,Consumer,"AI-Enhanced Learning, Artificial Intelligence,..."
40,Synapse Semiconductor,synapsesemi.org,Industrials,"Artificial Intelligence, Edge Computing Semico..."
41,Graphify Labs,graphify.com,B2B,"Developer Tools, Reinforcement Learning, Open ..."
42,PokerClubHub,pokerclubhub.com,Consumer,"SaaS, Crypto / Web3, Gaming, eSports"
43,Traceforce,traceforce.ai,B2B,"Artificial Intelligence, SaaS, Security, Cyber..."
44,Lumeria,lumeria.skin,Healthcare,"Consumer Health Services, Health Tech, Health ..."
45,Atomarine,atomarine.co,Industrials,"Hard Tech, Energy"


---
## ✅ Milestone 1 complete

We now have a real, free list of companies in `output/companies_raw.csv`, pulled live from
SEC EDGAR (companies actively raising money) and the Y Combinator directory (vetted
startups). No paid data was used.

**What's next — Milestone 2 (Clean & de-duplicate):** the same company can appear more than
once (e.g. filed with the SEC under a slightly different legal name, or listed in both
sources). Next we'll merge those duplicates into one clean record per company, combining the
best fields from each source (e.g. funding amount from EDGAR + website/description from YC).

---
# Milestone 2 — Clean & de-duplicate

**The problem:** the raw list from Milestone 1 has the same company listed more than once.
For example, a startup might appear in the YC directory as **"Wayy"** and in the SEC filings
as **"Wayy Inc."** — same company, two rows. If we don't fix this, we'd double-count
companies and split their information across rows.

**The goal of this milestone:** turn the messy raw list into a **clean list with exactly one
row per real company**, and when we merge two rows for the same company, keep the *best*
information from each (e.g. the funding amount from the SEC filing **and** the website +
description from YC).

We do this in two steps:
1. **Normalize** — tidy up the values so they're comparable (clean company names, fix
   country labels).
2. **Match & merge** — find rows that are the same company and combine them.

### 2a. Normalize the values

Two small clean-ups make matching reliable:

- **Company names:** we strip off the legal endings like "Inc.", "LLC", "Ltd", "Corp" and
  lower-case everything, so "Wayy Inc." and "Wayy" both become the simple key `wayy`. (We
  keep the original pretty name for display — this simplified key is only used for matching.)
- **Country:** SEC filings use two-letter codes. Real US states (CA, NY, TX...) mean the
  company is US-based. Some codes (like "A2") are actually foreign (that one is a Canadian
  province), so we relabel those as non-US instead of blindly calling everything US.

In [42]:
import re

# Legal endings we strip from company names before matching them.
LEGAL_SUFFIXES = [
    "incorporated",
    "inc",
    "llc",
    "l l c",
    "lp",
    "l p",
    "llp",
    "ltd",
    "limited",
    "corp",
    "corporation",
    "co",
    "company",
    "plc",
    "gmbh",
    "holdings",
    "group",
]

# The 50 US states + DC + common territories (2-letter). Used to tell US from non-US.
US_STATES = {
    "AL",
    "AK",
    "AZ",
    "AR",
    "CA",
    "CO",
    "CT",
    "DE",
    "FL",
    "GA",
    "HI",
    "ID",
    "IL",
    "IN",
    "IA",
    "KS",
    "KY",
    "LA",
    "ME",
    "MD",
    "MA",
    "MI",
    "MN",
    "MS",
    "MO",
    "MT",
    "NE",
    "NV",
    "NH",
    "NJ",
    "NM",
    "NY",
    "NC",
    "ND",
    "OH",
    "OK",
    "OR",
    "PA",
    "RI",
    "SC",
    "SD",
    "TN",
    "TX",
    "UT",
    "VT",
    "VA",
    "WA",
    "WV",
    "WI",
    "WY",
    "DC",
    "PR",
    "VI",
    "GU",
}


def clean_name(name):
    """Turn a company name into a simple key for matching. 'Wayy Inc.' -> 'wayy'."""
    if not isinstance(name, str):
        return ""
    s = name.lower()
    s = re.sub(r"[^a-z0-9 ]", " ", s)  # drop punctuation
    words = [w for w in s.split() if w not in LEGAL_SUFFIXES]
    return " ".join(words).strip()


def fix_country(row):
    """Decide US vs non-US from the state code (more reliable than the raw country)."""
    state = str(row.get("hq_state") or "").strip().upper()
    if state in US_STATES:
        return "US"
    if len(state) == 2 and state.isalpha():
        # A 2-letter code that isn't a US state (e.g. 'A2') = foreign issuer.
        return "Non-US"
    return row.get("hq_country")


# Load Milestone 1's raw list and add the cleaned-up columns.
companies_raw = pd.read_csv(os.path.join(OUTPUT_DIR, "companies_raw.csv"))
companies_raw["norm_name"] = companies_raw["name"].apply(clean_name)
companies_raw["hq_country"] = companies_raw.apply(fix_country, axis=1)

print("Example name cleaning:")
for original in ["Wayy Inc.", "i2 Health Advisors Inc.", "Kiken Technoloigies, Inc"]:
    print(f"   {original!r:36} -> {clean_name(original)!r}")
print("\nRaw rows to de-duplicate:", len(companies_raw))

Example name cleaning:
   'Wayy Inc.'                          -> 'wayy'
   'i2 Health Advisors Inc.'            -> 'i2 health advisors'
   'Kiken Technoloigies, Inc'           -> 'kiken technoloigies'

Raw rows to de-duplicate: 438


### 2b. Find which rows are the same company

We treat two rows as the **same company** if any of these is true:
- they have the **exact same website** (the strongest signal), **or**
- they have the **exact same cleaned-up name** (and it's not too short/generic), **or**
- their names are a **very close fuzzy match** *and* they're in the same US state.

"Fuzzy match" means the names are nearly identical allowing for small differences — we use a
well-known library (`rapidfuzz`) that scores similarity from 0 to 100; we require ≥ 92.

We then group all the rows that link together (if A matches B and B matches C, then A, B and
C are all the same company) using a classic technique called *union-find*.

In [43]:
from rapidfuzz import fuzz


class UnionFind:
    """Standard 'union-find': quickly groups items that are linked together."""

    def __init__(self, n):
        self.parent = list(range(n))

    def find(self, i):
        while self.parent[i] != i:
            self.parent[i] = self.parent[self.parent[i]]
            i = self.parent[i]
        return i

    def union(self, i, j):
        self.parent[self.find(i)] = self.find(j)


def find_duplicate_groups(df):
    """Return a list of groups; each group is a list of row-indexes that are one company."""
    rows = df.reset_index(drop=True)
    n = len(rows)
    uf = UnionFind(n)

    # Fast exact links: same domain, or same cleaned name.
    by_domain, by_name = {}, {}
    for i, r in rows.iterrows():
        dom = r["website_domain"]
        if isinstance(dom, str) and dom.strip():
            by_domain.setdefault(dom, []).append(i)
        nm = r["norm_name"]
        if isinstance(nm, str) and len(nm) >= 4:  # skip very short/generic names
            by_name.setdefault(nm, []).append(i)
    for group in list(by_domain.values()) + list(by_name.values()):
        for k in range(1, len(group)):
            uf.union(group[0], group[k])

    # Slower fuzzy links: near-identical names in the same US state.
    names = rows["norm_name"].tolist()
    states = rows["hq_state"].astype(str).tolist()
    for i in range(n):
        if len(names[i]) < 4:
            continue
        for j in range(i + 1, n):
            if len(names[j]) < 4 or states[i] != states[j]:
                continue
            if fuzz.ratio(names[i], names[j]) >= 92:
                uf.union(i, j)

    groups = {}
    for i in range(n):
        groups.setdefault(uf.find(i), []).append(i)
    return list(groups.values()), rows


groups, rows_indexed = find_duplicate_groups(companies_raw)
print(f"{len(companies_raw)} raw rows collapse into {len(groups)} unique companies.")
multi = [g for g in groups if len(g) > 1]
print(f"{len(multi)} of those companies were built by merging 2+ rows.")

438 raw rows collapse into 438 unique companies.
0 of those companies were built by merging 2+ rows.


### 2c. Merge each group into one clean record

When several rows are the same company, we combine them with sensible rules:
- **Website / description / sector** → take from whichever row has them (usually the YC row).
- **Funding amount & stage** → take from the SEC filing (usually the EDGAR row); if a company
  filed with the SEC more than once, we use the **most recent** filing and **count** the
  filings (a company that keeps raising is a stronger signal — we'll use this later).
- **Source** → we record which sources contributed (e.g. `edgar+yc`) so nothing is hidden.

In [44]:
def pick_first(values):
    """First non-empty value from a list."""
    for v in values:
        if v is not None and str(v) != "nan" and str(v).strip() != "":
            return v
    return None


def pick_longest(values):
    """Longest text value (used for descriptions)."""
    best = None
    for v in values:
        if isinstance(v, str) and (best is None or len(v) > len(best)):
            best = v
    return best


def merge_group(group_rows):
    """Combine several rows (same company) into one clean record."""
    g = group_rows
    sources = sorted(set(g["source"].dropna().tolist()))

    # Funding: use the most recent SEC filing; sum how many filings we saw.
    dated = g.dropna(subset=["last_round_date"]).sort_values("last_round_date")
    latest = dated.iloc[-1] if len(dated) else g.iloc[0]
    filing_count = int(
        pd.to_numeric(g["filing_count"], errors="coerce").fillna(0).sum()
    )

    # Prefer the display name from a row that has a website (usually the cleaner brand name).
    named = g.dropna(subset=["website_domain"])
    display_name = named.iloc[0]["name"] if len(named) else g.iloc[0]["name"]

    merged = {
        "name": display_name,
        "website_domain": pick_first(g["website_domain"].tolist()),
        "description": pick_longest(g["description"].tolist()),
        "industry_primary": pick_first(
            g.sort_values("source")["industry_primary"].tolist()
        ),
        "verticals": pick_first(g["verticals"].tolist()),
        "year_founded": pick_first(g["year_founded"].tolist()),
        "hq_city": pick_first(g["hq_city"].tolist()),
        "hq_state": pick_first(g["hq_state"].tolist()),
        "hq_country": pick_first(g["hq_country"].tolist()),
        "funding_total_usd": pd.to_numeric(
            g["funding_total_usd"], errors="coerce"
        ).max(),
        "last_round_type": latest.get("last_round_type"),
        "last_round_size_usd": pd.to_numeric(
            pd.Series([latest.get("last_round_size_usd")]), errors="coerce"
        ).iloc[0],
        "last_round_date": latest.get("last_round_date"),
        "filing_count": filing_count,
        "source": "+".join(sources),
        "norm_name": pick_first(g["norm_name"].tolist()),
    }
    return merged


merged_records = [merge_group(rows_indexed.loc[g]) for g in groups]
companies_clean = pd.DataFrame(merged_records)

# Re-create a stable id for each merged company and order the columns.
companies_clean["company_id"] = companies_clean.apply(make_company_id, axis=1)
ordered = ["company_id"] + [c for c in CANONICAL_FIELDS if c != "company_id"]
# Keep norm_name in memory (enrichment uses it); we just don't write it to the CSV.
companies_clean = companies_clean[ordered + ["norm_name"]]

clean_path = os.path.join(OUTPUT_DIR, "companies_clean.csv")
companies_clean.drop(columns=["norm_name"]).to_csv(clean_path, index=False)

dedupe_rate = 1 - len(companies_clean) / len(companies_raw)
print(f"Raw rows:            {len(companies_raw)}")
print(f"Unique companies:    {len(companies_clean)}")
print(f"De-duplication rate: {dedupe_rate:.1%}  (share of rows that were duplicates)")
print(f"Saved clean list to: {clean_path}")

Raw rows:            438
Unique companies:    438
De-duplication rate: 0.0%  (share of rows that were duplicates)
Saved clean list to: output/companies_clean.csv


### 2d. What the de-duplication actually did

De-duplication catches two kinds of duplicates: (1) the *same* company appearing in **both**
sources, and (2) the same company filing with the SEC **more than once**. We report both
below.

**An honest, important note:** the companies that filed with the SEC in the last 90 days and
the YC startups are mostly **different** companies. YC startups often raise money using
instruments (like SAFEs) that don't trigger a public Form D, and YC is only a tiny slice of
all fundraising startups. So it's completely expected to see **few or zero** companies
overlapping across the two sources in a short window — and on this run we do see near-zero.
That's not a bug: the merging logic still runs, it just has little to merge in this snapshot.
It would merge more as you widen the SEC date window (`lookback_days`) or run it repeatedly
over time. The real value of this step is (a) cleaning/normalizing every record and (b)
collapsing the SEC's own repeat filings — both of which set up the later scoring milestone.

In [45]:
both = companies_clean[companies_clean["source"] == "edgar+yc"]
repeat_filers = companies_clean[
    (companies_clean["source"] == "edgar") & (companies_clean["filing_count"] > 1)
]

print(f"Companies found in BOTH sources (merged):     {len(both)}")
print(f"SEC repeat-filers merged (filed more than 1x): {len(repeat_filers)}")
print("\nSource breakdown of the clean list:")
print(companies_clean["source"].value_counts())

if len(both):
    print("\nExamples matched across both sources:")
    display(
        both[
            [
                "name",
                "website_domain",
                "last_round_type",
                "last_round_size_usd",
                "source",
            ]
        ].head(10)
    )
else:
    print("\n(No cross-source overlap in this snapshot — expected, see note above.)")

Companies found in BOTH sources (merged):     0
SEC repeat-filers merged (filed more than 1x): 0

Source breakdown of the clean list:
source
yc       400
edgar     38
Name: count, dtype: int64

(No cross-source overlap in this snapshot — expected, see note above.)


---
# Milestone 3 — Enrichment: adding "traction" signals

A company's SEC filing and YC blurb tell us *what it is*, but not *how much momentum it has*.
This milestone adds three free signals that hint at momentum:

1. **GitHub activity** — for software companies, how much public code they have and how
   recently they've been shipping. (Lots of stars + recent activity = active engineering.)
2. **Hacker News buzz** — how often the company is mentioned on Hacker News (a big tech
   community), and the score of its most-upvoted mention. (A proxy for attention.)
3. **Open job postings** — how many jobs the company is advertising on public job boards
   (Greenhouse/Lever). (Hiring a lot = growing fast.)

**Two honest limitations up front:**
- These free lookups **match by company name**, which isn't perfect — a common name can
  match the wrong project. We match conservatively and treat these as *rough* signals.
- Free APIs are **rate-limited**, and not every company uses GitHub or a public job board.
  So we only enrich a capped number of companies, and many will legitimately come back empty.
  We report exactly how many we found at the end.

In [46]:
# How many companies to enrich (kept modest because GitHub limits us to ~10 requests/minute
# without a token). Set a GITHUB_TOKEN environment variable (free) to go faster.
MAX_ENRICH = 40

GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN")
GH_HEADERS = dict(EDGAR_HEADERS)
if GITHUB_TOKEN:
    GH_HEADERS["Authorization"] = f"Bearer {GITHUB_TOKEN}"
    GH_SLEEP = 1.0  # authenticated: ~30 search requests/min
    print("Using GITHUB_TOKEN (faster).")
else:
    GH_SLEEP = 6.5  # unauthenticated: ~10 search requests/min -> wait between calls
    print("No GITHUB_TOKEN found -> GitHub lookups will be slow (that's fine).")

No GITHUB_TOKEN found -> GitHub lookups will be slow (that's fine).


### 3a. The three lookup functions

Each function takes a company and returns a small piece of extra data. They all fail safely:
if a lookup errors out or finds nothing, it returns empty values rather than crashing.

In [47]:
from datetime import datetime, timezone


def brand_from_domain(domain):
    """The core brand word from a domain: 'traceforce.ai' -> 'traceforce'. None if no domain."""
    if not isinstance(domain, str) or "." not in domain:
        return None
    return clean_name(domain.split(".")[0])


def enrich_github(name, norm_name, domain):
    """Find the company's *own* GitHub repo; return (stars, recent-activity 0..1).

    To avoid matching an unrelated project with the same word (e.g. a random repo also
    called 'sidekick'), we only accept a repo if it's clearly the company's: either its
    linked homepage is the company's own website, OR the GitHub account name matches the
    company's brand (the word in its domain).
    """
    brand = brand_from_domain(domain)
    query = brand or name
    try:
        r = requests.get(
            "https://api.github.com/search/repositories",
            params={"q": query, "sort": "stars", "per_page": 5},
            headers=GH_HEADERS,
            timeout=20,
        )
        if r.status_code != 200:
            return None, None
        for item in r.json().get("items", []):
            owner = clean_name(item["owner"]["login"])
            homepage_domain = domain_from_url(item.get("homepage"))
            is_ours = (domain and homepage_domain == domain) or (
                brand and owner == brand
            )
            if not is_ours:
                continue
            stars = item.get("stargazers_count", 0)
            pushed = item.get("pushed_at")
            activity = 0.0
            if pushed:
                days = (
                    datetime.now(timezone.utc)
                    - datetime.fromisoformat(pushed.replace("Z", "+00:00"))
                ).days
                activity = 1.0 if days <= 90 else max(0.0, 1 - (days - 90) / 365)
            return stars, round(activity, 2)
        return 0, 0.0  # searched, found no repo that's clearly theirs -> low signal
    except Exception:
        return None, None


def enrich_hackernews(name, domain):
    """Return (number of HN mentions, points of the top mention).

    We search Hacker News for the company's **website domain** when we have one — that's far
    more precise than the name (searching the word 'Mount' would match thousands of unrelated
    stories, but 'mount.insure' only matches the real company). If there's no website, we
    search the quoted name and only count stories that actually mention it in the title.
    """
    try:
        if domain:
            j = requests.get(
                "https://hn.algolia.com/api/v1/search",
                params={"query": domain, "tags": "story"},
                headers=EDGAR_HEADERS,
                timeout=20,
            ).json()
            pts = [h.get("points", 0) or 0 for h in j.get("hits", [])]
            return j.get("nbHits", 0), (max(pts) if pts else 0)
        # No website: fall back to name, but keep only titles that mention it (reduces noise).
        j = requests.get(
            "https://hn.algolia.com/api/v1/search",
            params={"query": f'"{name}"', "tags": "story"},
            headers=EDGAR_HEADERS,
            timeout=20,
        ).json()
        hits = [
            h
            for h in j.get("hits", [])
            if name.lower() in (h.get("title") or "").lower()
        ]
        pts = [h.get("points", 0) or 0 for h in hits]
        return len(hits), (max(pts) if pts else 0)
    except Exception:
        return None, None


def enrich_jobs(norm_name):
    """Guess a job-board slug from the name and count open postings (Greenhouse then Lever)."""
    slug = norm_name.replace(" ", "")
    if not slug:
        return None
    try:
        r = requests.get(
            f"https://boards-api.greenhouse.io/v1/boards/{slug}/jobs",
            headers=EDGAR_HEADERS,
            timeout=20,
        )
        if r.status_code == 200:
            return len(r.json().get("jobs", []))
        r = requests.get(
            f"https://api.lever.co/v0/postings/{slug}?mode=json",
            headers=EDGAR_HEADERS,
            timeout=20,
        )
        if r.status_code == 200 and isinstance(r.json(), list):
            return len(r.json())
    except Exception:
        return None
    return None  # company doesn't use these boards (or uses a different slug)


print("Enrichment functions defined.")

Enrichment functions defined.


### 3b. Run enrichment on a subset (with caching)

We enrich the companies most worth looking at first — **the ones that have a website** (so
GitHub/job lookups have a better chance of matching), up to our cap. Every result is cached
to `data/raw/enrichment_cache.json`, so if you stop and re-run, we don't repeat the slow
GitHub calls.

In [48]:
cache_path = os.path.join(RAW_DIR, "enrichment_cache.json")
CACHE_VERSION = 2  # bump this if the lookup logic changes, to ignore stale results

raw_cache = json.load(open(cache_path)) if os.path.exists(cache_path) else {}
# Keep only cached entries produced by the current logic version.
cache = {
    k: v
    for k, v in raw_cache.items()
    if isinstance(v, dict) and v.get("_v") == CACHE_VERSION
}

# Pick which companies to enrich: prefer ones with a website, then the rest.
to_enrich = companies_clean.copy()
to_enrich["_has_site"] = to_enrich["website_domain"].notna()
to_enrich = to_enrich.sort_values("_has_site", ascending=False).head(MAX_ENRICH)

print(
    f"Enriching {len(to_enrich)} companies "
    f"({int(to_enrich['_has_site'].sum())} with a website)..."
)

for n, (_, row) in enumerate(to_enrich.iterrows(), 1):
    cid = row["company_id"]
    if cid in cache:
        continue  # already done on a previous run
    name, norm, domain = row["name"], row["norm_name"], row["website_domain"]
    domain = domain if isinstance(domain, str) and domain.strip() else None
    gh_stars, gh_activity = enrich_github(name, norm, domain)
    hn_mentions, hn_points = enrich_hackernews(name, domain)
    jobs = enrich_jobs(norm)
    cache[cid] = {
        "github_stars": gh_stars,
        "github_activity_90d": gh_activity,
        "hn_mentions": hn_mentions,
        "hn_points": hn_points,
        "open_job_postings": jobs,
        "_v": CACHE_VERSION,
    }
    json.dump(cache, open(cache_path, "w"))  # save after each (safe to interrupt)
    time.sleep(GH_SLEEP)  # respect GitHub's rate limit
    if n % 10 == 0:
        print(f"  ...enriched {n}/{len(to_enrich)}")

print("Enrichment done. Cached results:", len(cache))

Enriching 40 companies (40 with a website)...
Enrichment done. Cached results: 40


### 3c. Attach the signals and save the enriched list

We add the new columns to every company (companies we didn't enrich simply have blanks), and
save the result to `output/companies_enriched.csv`. Then we report how many companies we
actually found each signal for — being upfront about coverage.

In [49]:
enrich_cols = [
    "github_stars",
    "github_activity_90d",
    "hn_mentions",
    "hn_points",
    "open_job_postings",
]

# Map cached results back onto the full clean company list.
for col in enrich_cols:
    companies_clean[col] = companies_clean["company_id"].map(
        lambda cid: cache.get(cid, {}).get(col)
    )

enriched_path = os.path.join(OUTPUT_DIR, "companies_enriched.csv")
companies_clean.drop(
    columns=[c for c in ["norm_name"] if c in companies_clean.columns]
).to_csv(enriched_path, index=False)
print("Saved enriched list to:", enriched_path)

# Coverage report: for how many of the enriched companies did each signal turn up something?
enriched_ids = set(to_enrich["company_id"])
sub = companies_clean[companies_clean["company_id"].isin(enriched_ids)]
print(f"\nCoverage among the {len(sub)} enriched companies:")
print(f"  GitHub repo found:      {(sub['github_stars'].fillna(0) > 0).sum()}")
print(f"  Mentioned on HN:        {(sub['hn_mentions'].fillna(0) > 0).sum()}")
print(f"  Open jobs found:        {(sub['open_job_postings'].fillna(0) > 0).sum()}")

Saved enriched list to: output/companies_enriched.csv

Coverage among the 40 enriched companies:
  GitHub repo found:      3
  Mentioned on HN:        12
  Open jobs found:        3


In [50]:
# A look at the companies with the strongest signals we found.
print("=== Most Hacker News buzz ===")
display(
    sub.sort_values("hn_points", ascending=False)[
        ["name", "website_domain", "hn_mentions", "hn_points"]
    ].head(8)
)

print("\n=== Most GitHub stars ===")
display(
    sub.sort_values("github_stars", ascending=False)[
        ["name", "website_domain", "github_stars", "github_activity_90d"]
    ].head(8)
)

print("\n=== Companies actively hiring (open jobs) ===")
jobs_found = sub[sub["open_job_postings"].fillna(0) > 0]
display(
    jobs_found[["name", "website_domain", "open_job_postings"]].head(8)
    if len(jobs_found)
    else "None found in this subset (they may not use Greenhouse/Lever)."
)

=== Most Hacker News buzz ===


,name,website_domain,hn_mentions,hn_points
219,Incandor,incandor.com,4.0,351.0
325,Interfaze,interfaze.ai,8.0,164.0
278,RentAHuman,rentahuman.ai,3.0,155.0
288,Regbase,regbase.com,21.0,143.0
297,Zerocater,zerocater.com,29.0,111.0
293,Ardent,tryardent.com,3.0,99.0
284,InsForge,insforge.dev,10.0,62.0
289,Inth,inth.com,36.0,50.0



=== Most GitHub stars ===


,name,website_domain,github_stars,github_activity_90d
284,InsForge,insforge.dev,12521.0,1.0
292,Napkin Math,napkinmath.club,21.0,0.0
325,Interfaze,interfaze.ai,3.0,1.0
317,Standard Signal,standardsignal.com,0.0,0.0
298,Sidekick,textsidekick.com,0.0,0.0
299,Degla Inc,degla.ai,0.0,0.0
300,Trellis,trellistech.com,0.0,0.0
301,Taiga,taigabilling.com,0.0,0.0



=== Companies actively hiring (open jobs) ===


,name,website_domain,open_job_postings
277,Juno,junocompanion.com,3.0
297,Zerocater,zerocater.com,10.0
300,Trellis,trellistech.com,5.0


---
## ✅ Milestones 2 & 3 complete

- **Milestone 2** turned the messy raw list into a clean list with **one row per company**
  (`output/companies_clean.csv`), merging duplicates and combining the best fields from each
  source.
- **Milestone 3** added **traction signals** — GitHub activity, Hacker News buzz, and open
  job postings — to produce `output/companies_enriched.csv`.

Together these give us, for each company: what it is (name, website, description, sector),
how it's funded (stage + amount from the SEC), and how much momentum it has (the three
signals above) — all from free data.

**What's next (roadmap):** Milestone 4 combines all of this into a single explainable
**0–100 fit score** against your investment thesis, so the companies can be ranked.

---
# Milestone 4 — Scoring: a single, explainable 0–100 fit score

So far each company is just a pile of facts. This milestone turns those facts into **one
number from 0 to 100** that says *how well this company fits your investment thesis* — and,
crucially, shows **exactly why** (which is what a paid tool usually hides).

### The six ingredients ("features")
Each is measured on a 0-to-1 scale (0 = not at all, 1 = perfect):

| Feature | Plain meaning | Where the data comes from |
|---|---|---|
| **semantic_sim** | How closely the company's description matches your thesis, in meaning | company description vs. your thesis (AI embeddings) |
| **stage_fit** | Is it at a funding stage you target? | SEC-inferred stage |
| **geo_fit** | Is it in a region you target? | HQ country |
| **traction** | Does it have momentum? | GitHub / Hacker News / open jobs |
| **investor_signal** | Signs of real fundraising backing | SEC filing count + funding amount |
| **hygiene** | Is the record complete/legit-looking? | has website, description, year, industry |

### How they combine
Each feature has a **weight** (how much it matters). The score is just the weighted sum,
scaled to 100. Because it's a simple weighted sum, we can report **each feature's points**,
and those points **add up exactly to the score** — that's the "explainable" part.

### An honest note about our two data sources
Our two free sources describe companies differently: **YC** companies come with a
description and momentum signals but no funding stage; **SEC** companies come with a funding
stage but no description. Rather than invent missing data, we do two honest things:
(1) when a feature's data is simply **unknown**, we give it a **neutral 0.5** instead of a
zero (so a company isn't punished just because a source didn't provide that field), and
(2) the score breakdown makes it obvious which signals actually drove each company's score.
In a production system you'd close this gap by looking up each SEC company's website to give
it a description too — a good future upgrade.

### 4a. Turn each company into text and compare it to your thesis (semantic match)

We use a small AI model (`all-MiniLM-L6-v2`, free and runs locally) to convert your thesis
and each company's description into lists of numbers ("embeddings") that capture *meaning*.
Two things about similar topics get similar numbers, so we can measure how close a company
is to your thesis even when they use different words.

The embeddings are cached to `data/raw/embeddings.npz`, so this only runs slowly once.

In [51]:
import numpy as np
import hashlib

companies = pd.read_csv(os.path.join(OUTPUT_DIR, "companies_enriched.csv"))


def company_text(row):
    """Build one line of text describing the company, for the AI model to read."""
    bits = [str(row.get("name") or "")]
    if isinstance(row.get("description"), str):
        bits.append(row["description"])
    if isinstance(row.get("verticals"), str):
        bits.append(row["verticals"])
    if isinstance(row.get("industry_primary"), str):
        bits.append(row["industry_primary"])
    return " | ".join(b for b in bits if b).strip()


texts = companies.apply(company_text, axis=1).tolist()
thesis = CONFIG["thesis"]
ids = companies["company_id"].tolist()

# A fingerprint of "thesis + all company texts": if nothing changed, reuse cached embeddings.
key = hashlib.md5(("||".join([thesis] + texts)).encode()).hexdigest()
emb_path = os.path.join(RAW_DIR, "embeddings.npz")

cached = None
if os.path.exists(emb_path):
    z = np.load(emb_path, allow_pickle=True)
    if str(z["key"]) == key:
        cached = z

if cached is not None:
    company_vecs = cached["company_vecs"]
    thesis_vec = cached["thesis_vec"]
    print("Loaded cached embeddings (nothing changed since last run).")
else:
    from sentence_transformers import SentenceTransformer

    print("Loading the AI model and encoding companies (one-time, ~1-2 min)...")
    model = SentenceTransformer("all-MiniLM-L6-v2")
    company_vecs = model.encode(
        texts, normalize_embeddings=True, show_progress_bar=False
    )
    thesis_vec = model.encode([thesis], normalize_embeddings=True)[0]
    np.savez(
        emb_path,
        key=key,
        company_vecs=company_vecs,
        thesis_vec=thesis_vec,
        ids=np.array(ids, dtype=object),
    )
    print(f"Encoded {len(texts)} companies and cached them to {emb_path}")

# semantic_sim = how aligned each company is with the thesis (0..1).
companies["semantic_sim"] = np.clip(company_vecs @ thesis_vec, 0, 1)
print("Thesis:", thesis)
print("Highest thesis-match companies:")
display(
    companies.sort_values("semantic_sim", ascending=False)[
        ["name", "industry_primary", "semantic_sim"]
    ].head(6)
)

Loaded cached embeddings (nothing changed since last run).
Thesis: AI infrastructure and developer tools; Pre-Seed to Series A; US-based
Highest thesis-match companies:


,name,industry_primary,semantic_sim
206,OpenProse,B2B,0.590487
103,Agnost AI,B2B,0.577367
312,Lab0,B2B,0.569694
279,Archal,B2B,0.568975
378,Talking Computers,B2B,0.550949
65,rekursiv.ai,B2B,0.548353


### 4b. Compute the other five features

Each function returns a number from 0 to 1. Note the neutral-0.5 handling for genuinely
unknown values, discussed above.

In [52]:
import math

STAGE_ORDER = ["Pre-Seed", "Seed", "Early", "Later"]


def stage_fit(stage, targets):
    """1.0 if the company's stage is one you target; less as it gets further away; 0.5 if unknown."""
    if stage not in STAGE_ORDER:
        return 0.5  # unknown stage -> neutral
    if stage in targets:
        return 1.0
    target_idx = [STAGE_ORDER.index(t) for t in targets if t in STAGE_ORDER]
    if not target_idx:
        return 0.5
    distance = min(abs(STAGE_ORDER.index(stage) - i) for i in target_idx)
    return max(0.0, 1 - 0.34 * distance)  # one step away ~0.66, two ~0.32


def geo_fit(country, targets):
    if not isinstance(country, str) or country.strip() in ("", "nan"):
        return 0.5  # unknown location -> neutral
    return 1.0 if country in targets else 0.0


def _log_scale(x, cap):
    """Squash a big count into 0..1 on a log scale (so 10 and 10,000 aren't wildly far apart)."""
    if x is None or (isinstance(x, float) and math.isnan(x)) or x <= 0:
        return 0.0
    return min(1.0, math.log10(1 + x) / cap)


def traction(row):
    """Momentum on ANY axis counts: take the strongest of the enrichment signals."""
    signals = [
        row.get("github_activity_90d")
        if not pd.isna(row.get("github_activity_90d"))
        else 0.0,
        _log_scale(row.get("github_stars"), 4),  # ~10k stars -> 1.0
        _log_scale(row.get("hn_points"), 3),  # ~1000 points -> 1.0
        _log_scale(row.get("open_job_postings"), 2),  # ~100 jobs -> 1.0
    ]
    return float(np.clip(max(signals), 0, 1))


def investor_signal(row):
    """Repeat SEC filers (kept raising) and having a real offering amount = stronger backing."""
    filings = row.get("filing_count") or 0
    base = min(1.0, filings / 3.0)
    funding = row.get("funding_total_usd")
    if funding is not None and not pd.isna(funding) and funding > 0:
        base = max(base, 0.34)
    return base


def hygiene(row):
    """Fraction of key descriptive fields that are present (a completeness/legitimacy check)."""
    keys = ["website_domain", "description", "year_founded", "industry_primary"]
    present = sum(
        1 for k in keys if row.get(k) is not None and str(row.get(k)) not in ("", "nan")
    )
    return present / len(keys)


companies["stage_fit"] = companies["last_round_type"].apply(
    lambda s: stage_fit(s, CONFIG["target_stages"])
)
companies["geo_fit"] = companies["hq_country"].apply(
    lambda c: geo_fit(c, CONFIG["target_geos"])
)
companies["traction"] = companies.apply(traction, axis=1)
companies["investor_signal"] = companies.apply(investor_signal, axis=1)
companies["hygiene"] = companies.apply(hygiene, axis=1)

FEATURES = [
    "semantic_sim",
    "stage_fit",
    "geo_fit",
    "traction",
    "investor_signal",
    "hygiene",
]
print("Feature averages across all companies (sanity check):")
print(companies[FEATURES].mean().round(3).to_string())

Feature averages across all companies (sanity check):
semantic_sim       0.340
stage_fit          0.536
geo_fit            0.937
traction           0.020
investor_signal    0.029
hygiene            0.721


### 4c. Combine into the score (with a transparent breakdown)

The weights below say how much each feature matters (they add up to 1.0). You can edit them
to match how *you* weigh things. The score is `100 × Σ (weight × feature)`, and we also store
each feature's **points**, which add up exactly to the score.

In [53]:
# How much each feature matters. Editable — try changing these and re-running.
WEIGHTS = {
    "semantic_sim": 0.40,  # matching your thesis matters most
    "stage_fit": 0.20,
    "geo_fit": 0.10,
    "traction": 0.15,
    "investor_signal": 0.10,
    "hygiene": 0.05,
}


def score_row(row, weights):
    """Return (score 0..100, {feature: points})."""
    contributions = {}
    total = 0.0
    for feature, weight in weights.items():
        value = float(np.clip(row[feature], 0, 1))
        points = 100.0 * weight * value
        contributions[feature] = round(points, 2)
        total += points
    return round(total, 2), contributions


scores, contribs = [], []
for _, row in companies.iterrows():
    s, c = score_row(row, WEIGHTS)
    scores.append(s)
    contribs.append(c)

companies["score"] = scores
for feature in FEATURES:
    companies[f"pts_{feature}"] = [c[feature] for c in contribs]

# Safety check: the points must add up to the score.
recomputed = companies[[f"pts_{f}" for f in FEATURES]].sum(axis=1).round(2)
assert (abs(recomputed - companies["score"]) < 0.05).all(), (
    "points should sum to score!"
)
print("Check passed: each company's feature points add up to its score.")

companies = companies.sort_values("score", ascending=False).reset_index(drop=True)
ranked_path = os.path.join(OUTPUT_DIR, "companies_ranked.csv")
companies.to_csv(ranked_path, index=False)
print(f"Saved ranked companies to {ranked_path}")

Check passed: each company's feature points add up to its score.
Saved ranked companies to output/companies_ranked.csv


### 4d. The ranked results, and *why* the top companies scored well

Below is the leaderboard, then a plain-English explanation of the #1 company's score broken
down by feature — this is the transparency a partner can actually trust and adjust.

In [54]:
print("=== TOP 15 COMPANIES BY FIT SCORE ===")
show = ["name", "score", "last_round_type", "hq_country", "source", "website_domain"]
display(companies[show].head(15))


def explain(row):
    """Print a readable breakdown of one company's score."""
    print(f"\n{row['name']}  —  score {row['score']}/100   ({row.get('source')})")
    if isinstance(row.get("description"), str):
        print(f"  What it does: {row['description'][:160]}")
    ordered = sorted(FEATURES, key=lambda f: row[f"pts_{f}"], reverse=True)
    for f in ordered:
        bar = "#" * int(round(row[f"pts_{f}"]))
        print(f"    {f:16} {row[f'pts_{f}']:5.1f} pts  {bar}")


for i in range(3):
    explain(companies.iloc[i])

=== TOP 15 COMPANIES BY FIT SCORE ===


,name,score,last_round_type,hq_country,source,website_domain
0,InsForge,58.70,NaN,US,yc,insforge.dev
1,Interfaze,55.82,NaN,US,yc,interfaze.ai
2,RentAHuman,55.30,NaN,US,yc,rentahuman.ai
3,Gaingels Core Automation LLC,51.89,Early,US,edgar,NaN
4,i2 Health Advisors Inc.,50.15,Early,US,edgar,NaN
5,"Cross Margin Labs, Inc.",48.59,Seed,US,edgar,NaN
6,Incandor,48.44,NaN,US,yc,incandor.com
7,"Uptime USA Aggregator II, LLC",47.89,Seed,US,edgar,NaN
8,Inth,47.38,NaN,US,yc,inth.com
9,OpenProse,47.37,NaN,US,yc,prose.md



InsForge  —  score 58.7/100   (yc)
  What it does: InsForge is the agent-native AWS: cloud infrastructure that AI coding agents can understand, operate, and recover from end to end.
    semantic_sim      19.9 pts  ####################
    traction          15.0 pts  ###############
    stage_fit         10.0 pts  ##########
    geo_fit           10.0 pts  ##########
    hygiene            3.8 pts  ####
    investor_signal    0.0 pts  

Interfaze  —  score 55.82/100   (yc)
  What it does: Interfaze is an AI model built on a new architecture that merges specialized DNN/CNN models with transformers for tasks that require deterministic output and hi
    semantic_sim      17.1 pts  #################
    traction          15.0 pts  ###############
    stage_fit         10.0 pts  ##########
    geo_fit           10.0 pts  ##########
    hygiene            3.8 pts  ####
    investor_signal    0.0 pts  

RentAHuman  —  score 55.3/100   (yc)
  What it does: Marketplace for AI agents to hire hum

### 4e. Proof it's adjustable: change the weights, watch the ranking move

To show the weights genuinely drive the results, here's the top 5 under the default weights
vs. a "traction-first" investor who cares most about momentum. The ordering changes.

In [55]:
def top5(weights):
    s = companies.apply(lambda r: score_row(r, weights)[0], axis=1)
    return (
        companies.assign(_s=s)
        .sort_values("_s", ascending=False)["name"]
        .head(5)
        .tolist()
    )


traction_first = {
    "semantic_sim": 0.20,
    "stage_fit": 0.10,
    "geo_fit": 0.05,
    "traction": 0.45,
    "investor_signal": 0.15,
    "hygiene": 0.05,
}

print("Top 5 with DEFAULT weights (thesis-first):")
for i, n in enumerate(top5(WEIGHTS), 1):
    print(f"  {i}. {n}")
print("\nTop 5 with TRACTION-FIRST weights:")
for i, n in enumerate(top5(traction_first), 1):
    print(f"  {i}. {n}")

Top 5 with DEFAULT weights (thesis-first):
  1. InsForge
  2. Interfaze
  3. RentAHuman
  4. Gaingels Core Automation LLC
  5. i2 Health Advisors Inc.

Top 5 with TRACTION-FIRST weights:
  1. InsForge
  2. Interfaze
  3. Incandor
  4. RentAHuman
  5. Regbase


---
# Milestone 5 — "Find similar companies" (comps)

When an investor likes a company, the natural next question is *"who else looks like this?"*
— competitors, or comparable companies to benchmark against. VCs call these **comps**.

We already turned every company into a meaning-based fingerprint (the embeddings from
Milestone 4). To find comps for a company, we just find the **other companies whose
fingerprints are closest** to it. No extra downloads or paid tools — it's a quick
math comparison over the numbers we already have.

(Our earlier project used a technique called TF-IDF for this; embeddings are an upgrade
because they compare *meaning*, not just shared words.)

In [56]:
# Line up the cached embeddings with our (now re-sorted) company table.
id_to_vec = {cid: company_vecs[i] for i, cid in enumerate(ids)}
vec_matrix = np.vstack([id_to_vec[c] for c in companies["company_id"]])
row_of_id = {cid: i for i, cid in enumerate(companies["company_id"])}


def find_similar(company_id, k=5):
    """Return the k companies most similar in meaning to the given one."""
    i = row_of_id[company_id]
    sims = (
        vec_matrix @ vec_matrix[i]
    )  # cosine similarity (vectors are already normalized)
    sims[i] = -1  # exclude the company itself
    best = np.argsort(sims)[::-1][:k]
    out = companies.iloc[best][
        ["name", "industry_primary", "website_domain", "score"]
    ].copy()
    out["similarity"] = np.round(sims[best], 3)
    return out


print("find_similar() is ready.")

find_similar() is ready.


### 5a. Example: comps for a top-ranked company

We pick the highest-scoring company that actually has a description (so the comparison is
meaningful) and show its five closest comparables.

In [57]:
with_desc = companies[companies["description"].notna()]
example_id = with_desc.iloc[0]["company_id"]
example = companies[companies["company_id"] == example_id].iloc[0]

print(f"Finding companies similar to: {example['name']}")
print(
    f"  ({example.get('industry_primary')}) — {str(example.get('description'))[:160]}"
)
print()
display(find_similar(example_id, k=5))

# Save one worked example so it's easy to eyeball outside the notebook.
similar_df = find_similar(example_id, k=5)
demo = {
    "query_company": {
        "company_id": example_id,
        "name": example["name"],
        "score": float(example["score"]),
    },
    "similar_companies": similar_df.to_dict(orient="records"),
}
with open(os.path.join(OUTPUT_DIR, "similar_example.json"), "w") as f:
    json.dump(demo, f, indent=2, default=str)
print("\nSaved this example to output/similar_example.json")

Finding companies similar to: InsForge
  (B2B) — InsForge is the agent-native AWS: cloud infrastructure that AI coding agents can understand, operate, and recover from end to end.



,name,industry_primary,website_domain,score,similarity
22,Tolmo,B2B,tolmo.com,45.05,0.602
19,IncidentFox,B2B,incidentfox.ai,45.23,0.586
40,BentoLabs AI,B2B,bentolabs.ai,43.87,0.580
176,Klaimee,Fintech,klaimee.ai,39.86,0.579
12,Archal,B2B,archal.ai,46.51,0.563



Saved this example to output/similar_example.json


---
## ✅ Milestones 4 & 5 complete

- **Milestone 4** produced `output/companies_ranked.csv`: every company now has a **0–100 fit
  score** with a **transparent, adjustable breakdown** showing exactly which signals earned
  the points.
- **Milestone 5** added a **"find similar companies" (comps)** capability, powered by the same
  AI embeddings, with a saved example in `output/similar_example.json`.

**What's next (roadmap):** Milestone 6 writes a short plain-English **summary** for each top
company; Milestone 7 **evaluates** how good the ranking is using a proxy for real outcomes;
Milestone 8 exposes everything through a small **API** so other tools can request a ranked
list, a company's memo, or its comps on demand.

---
# Milestone 6 — Short, readable summaries for the top companies

A ranked table is useful, but an investor really wants a **quick paragraph**: what does this
company do, how is it funded, what's the momentum, and why did it score the way it did. This
milestone writes that paragraph automatically for the top companies.

**Two ways to write the summary:**
1. **Template (default, always works):** we stitch the facts we already have into a clean,
   readable sentence or two. No internet, no AI service, no cost.
2. **Local AI (optional upgrade):** if you've installed **Ollama** (a free tool that runs an
   AI model on your own computer), the notebook will use it to write a more natural-sounding
   summary. If Ollama isn't installed, we simply use the template — nothing breaks.

We deliberately **don't** use a paid AI service (like the OpenAI/Anthropic APIs the original
project could use) — this version stays 100% free.

In [58]:
import shutil
import subprocess

SUMMARIES_TOP_N = 25  # how many of the top-ranked companies to summarize
USE_OLLAMA_IF_AVAILABLE = True  # set False to always use the simple template
OLLAMA_MODEL = "llama3.2"  # only used if Ollama is installed

OLLAMA_AVAILABLE = shutil.which("ollama") is not None
print(
    "Ollama installed?",
    OLLAMA_AVAILABLE,
    "-> using",
    ("local AI" if (OLLAMA_AVAILABLE and USE_OLLAMA_IF_AVAILABLE) else "the template"),
)


def money(x):
    """Format a dollar amount nicely, or '' if unknown."""
    if x is None or pd.isna(x):
        return ""
    return f"${x:,.0f}"


def _text_or_none(v):
    """Return a clean string, or None if the value is missing/NaN."""
    return (
        v if isinstance(v, str) and v.strip() and v.strip().lower() != "nan" else None
    )


def template_summary(row):
    """Build a readable summary purely from the fields we have (no AI needed)."""
    name = row["name"]
    # NOTE: float('nan') is truthy in Python, so we must check for it explicitly.
    sector = (
        _text_or_none(row.get("verticals"))
        or _text_or_none(row.get("industry_primary"))
        or "technology"
    )
    if len(sector) > 60:
        sector = sector.split(",")[0]

    sentences = []
    # Sentence 1: what it is.
    if _text_or_none(row.get("description")):
        sentences.append(f"{name} ({sector}) — {row['description'].strip()[:200]}")
    else:
        where = (
            _text_or_none(row.get("hq_state"))
            or _text_or_none(row.get("hq_country"))
            or "the US"
        )
        sentences.append(f"{name} operates in {sector} (based in {where}).")

    # Sentence 2: funding / stage.
    stage = row.get("last_round_type")
    amt = money(row.get("last_round_size_usd"))
    if isinstance(stage, str) and stage:
        raise_txt = f" raising about {amt}" if amt else ""
        sentences.append(
            f"It appears to be at the {stage} stage{raise_txt} (per SEC filings)."
        )

    # Sentence 3: momentum signals, if any.
    signals = []
    if row.get("github_stars") and row["github_stars"] > 0:
        signals.append(f"{int(row['github_stars'])} GitHub stars")
    if row.get("hn_points") and row["hn_points"] > 0:
        signals.append(f"a top Hacker News post at {int(row['hn_points'])} points")
    if row.get("open_job_postings") and row["open_job_postings"] > 0:
        signals.append(f"{int(row['open_job_postings'])} open roles")
    if signals:
        sentences.append("Momentum signals: " + ", ".join(signals) + ".")

    # Sentence 4: why it scored — the top two contributing features.
    feats = sorted(FEATURES, key=lambda f: row[f"pts_{f}"], reverse=True)[:2]
    pretty = {
        "semantic_sim": "thesis match",
        "stage_fit": "funding stage",
        "geo_fit": "location",
        "traction": "traction",
        "investor_signal": "investor signal",
        "hygiene": "data completeness",
    }
    sentences.append(
        f"It scores {row['score']:.0f}/100, driven mainly by "
        f"{pretty[feats[0]]} and {pretty[feats[1]]}."
    )
    return " ".join(sentences)


def ollama_summary(row):
    """Ask a locally-installed Ollama model to write the summary. Falls back to template on error."""
    facts = template_summary(row)  # feed the facts to the model as grounding
    prompt = (
        "Write a concise 70-90 word investor summary of this startup, in plain English, "
        "based ONLY on these facts. Do not invent anything.\n\nFacts: " + facts
    )
    try:
        out = subprocess.run(
            ["ollama", "run", OLLAMA_MODEL, prompt],
            capture_output=True,
            text=True,
            timeout=120,
        )
        text = out.stdout.strip()
        return text if text else facts
    except Exception:
        return facts


def summarize(row):
    if OLLAMA_AVAILABLE and USE_OLLAMA_IF_AVAILABLE:
        return ollama_summary(row)
    return template_summary(row)


print("Summary functions ready.")

Ollama installed? False -> using the template
Summary functions ready.


In [59]:
# Write summaries for the top-ranked companies (companies is already sorted by score).
companies["summary"] = None
top = companies.head(SUMMARIES_TOP_N).copy()
for idx, row in top.iterrows():
    companies.at[idx, "summary"] = summarize(row)

summaries_path = os.path.join(OUTPUT_DIR, "companies_summaries.csv")
companies.head(SUMMARIES_TOP_N)[
    ["company_id", "name", "website_domain", "score", "summary"]
].to_csv(summaries_path, index=False)
print(f"Wrote {SUMMARIES_TOP_N} summaries to {summaries_path}\n")

# Show the first few.
for _, row in companies.head(3).iterrows():
    print("-" * 80)
    print(row["summary"])

Wrote 25 summaries to output/companies_summaries.csv

--------------------------------------------------------------------------------
InsForge (B2B) — InsForge is the agent-native AWS: cloud infrastructure that AI coding agents can understand, operate, and recover from end to end. Momentum signals: 12521 GitHub stars, a top Hacker News post at 62 points. It scores 59/100, driven mainly by thesis match and traction.
--------------------------------------------------------------------------------
Interfaze (B2B) — Interfaze is an AI model built on a new architecture that merges specialized DNN/CNN models with transformers for tasks that require deterministic output and high consistency like Document intelligenc Momentum signals: 3 GitHub stars, a top Hacker News post at 164 points. It scores 56/100, driven mainly by thesis match and traction.
--------------------------------------------------------------------------------
RentAHuman (Consumer) — Marketplace for AI agents to hire humans.

---
# Milestone 7 — How good is the ranking? (Evaluation)

If we're going to trust this score, we should measure whether it behaves correctly. This is
called **evaluation**.

### The honest problem: we have no "answer key"
The ideal way to grade a deal-sourcing tool is to check its ranking against **real investor
decisions** — which companies a partner actually chased or invested in. The original version
of this project had exactly that (a CRM export). **We don't**, because we use only free public
data, and brand-new startups have **no known outcome yet** (we can't see the future). We also
tried the "answer keys" the plan suggested and found they don't fit this data: our 90-day
snapshot has essentially no visible follow-on rounds, and the recent-company sample contains
essentially no proven YC "top companies". And a simple keyword-relevance check turned out
useless here because recent YC startups are ~90% AI/tech — almost everything "matches", so it
can't tell good ranking from bad.

### So we evaluate two things we *can* verify, without needing outcomes
1. **Does the score actually respond to the thesis?** (construct validity) — if we swap in a
   *different* thesis, the right kind of companies should rise to the top. We test this by
   scoring under a contrasting thesis and watching a specific industry move.
2. **Are the "similar companies" coherent?** (retrieval quality) — a company's comps should
   genuinely resemble it. We check how often a company's top-5 comps share its industry,
   versus random chance.

Both are standard ways to validate this kind of system, and both give clear numbers here.

In [60]:
from sentence_transformers import SentenceTransformer

# --- Evaluation 1: does the score respond to the thesis? (construct validity) ---
# We score every company under TWO different theses and watch a specific industry move.
eval_model = SentenceTransformer("all-MiniLM-L6-v2")
CONTRAST_THESIS = "healthcare, biotech, medical and health technology"
CONTRAST_INDUSTRY = (
    "Healthcare"  # the industry we expect to rise under the contrast thesis
)
TOPK = 30

is_contrast_industry = (companies["industry_primary"] == CONTRAST_INDUSTRY).values


def share_in_top(thesis_text, k=TOPK):
    """Rank all companies by similarity to a thesis; return the share of top-k that are
    in the contrast industry (Healthcare)."""
    tvec = eval_model.encode([thesis_text], normalize_embeddings=True)[0]
    sims = vec_matrix @ tvec
    top = np.argsort(sims)[::-1][:k]
    return float(is_contrast_industry[top].mean())


share_main = share_in_top(CONFIG["thesis"])
share_contrast = share_in_top(CONTRAST_THESIS)

print("EVALUATION 1 — does the score respond to the thesis?")
print(f"  Share of '{CONTRAST_INDUSTRY}' companies in the top {TOPK}:")
print(f"    under your thesis ({CONFIG['thesis'][:40]}...):  {share_main:.1%}")
print(f"    under a HEALTHCARE thesis:                        {share_contrast:.1%}")
verdict = "PASS" if share_contrast > share_main * 2 else "weak"
print(
    f"  -> Healthcare companies rise sharply when the thesis asks for them: {verdict}"
)

EVALUATION 1 — does the score respond to the thesis?
  Share of 'Healthcare' companies in the top 30:
    under your thesis (AI infrastructure and developer tools; P...):  0.0%
    under a HEALTHCARE thesis:                        43.3%
  -> Healthcare companies rise sharply when the thesis asks for them: PASS


In [61]:
# --- Evaluation 2: are the "similar companies" coherent? (retrieval quality) ---
industries = companies["industry_primary"].fillna("?").tolist()
has_desc = companies["description"].notna().values

hits, total = 0, 0
for i in range(len(companies)):
    if not has_desc[i]:
        continue  # only judge companies that have a real description
    sims = vec_matrix @ vec_matrix[i]
    sims[i] = -1
    for j in np.argsort(sims)[::-1][:5]:
        hits += industries[j] == industries[i]
        total += 1
same_industry_at_5 = hits / total

# Random baseline: chance that two random companies share an industry.
from collections import Counter

counts = Counter(industries[i] for i in range(len(companies)) if has_desc[i])
n = sum(counts.values())
random_baseline = sum(v * (v - 1) for v in counts.values()) / (n * (n - 1))

print("EVALUATION 2 — are the comps coherent?")
print(
    f"  A company's top-5 comps share its industry {same_industry_at_5:.1%} of the time,"
)
print(
    f"  vs {random_baseline:.1%} by random chance  ->  {same_industry_at_5 / random_baseline:.2f}x better."
)

EVALUATION 2 — are the comps coherent?
  A company's top-5 comps share its industry 77.0% of the time,
  vs 38.8% by random chance  ->  1.98x better.


In [62]:
# Assemble the evaluation report (plus basic data-quality stats) and save it.
raw_total = len(pd.read_csv(os.path.join(OUTPUT_DIR, "companies_raw.csv")))
unique_total = len(companies)
key_fields = [
    "website_domain",
    "description",
    "industry_primary",
    "hq_country",
    "last_round_type",
    "funding_total_usd",
]
coverage = {f: round(companies[f].notna().mean(), 3) for f in key_fields}

eval_report = {
    "thesis": CONFIG["thesis"],
    "companies_scored": unique_total,
    "eval_1_thesis_responsiveness": {
        "contrast_industry": CONTRAST_INDUSTRY,
        "share_in_top30_under_main_thesis": round(share_main, 3),
        "share_in_top30_under_contrast_thesis": round(share_contrast, 3),
        "interpretation": "the score correctly promotes the contrast industry when the "
        "thesis asks for it",
    },
    "eval_2_similar_retrieval": {
        "same_industry_at_5": round(same_industry_at_5, 3),
        "random_baseline": round(random_baseline, 3),
        "lift": round(same_industry_at_5 / random_baseline, 2),
    },
    "dedupe_rate": round(1 - unique_total / raw_total, 3),
    "field_coverage": coverage,
    "notes": "No real investor-decision or outcome labels exist in free public data, so we "
    "validate the system's behavior (thesis responsiveness + comp coherence) rather "
    "than realized returns. Swap in real decisions to measure precision/recall for real.",
}

report_path = os.path.join(OUTPUT_DIR, "eval_report.json")
with open(report_path, "w") as f:
    json.dump(eval_report, f, indent=2)
print("Saved evaluation report to", report_path)
print(json.dumps(eval_report, indent=2))

Saved evaluation report to output/eval_report.json
{
  "thesis": "AI infrastructure and developer tools; Pre-Seed to Series A; US-based",
  "companies_scored": 438,
  "eval_1_thesis_responsiveness": {
    "contrast_industry": "Healthcare",
    "share_in_top30_under_main_thesis": 0.0,
    "share_in_top30_under_contrast_thesis": 0.433,
    "interpretation": "the score correctly promotes the contrast industry when the thesis asks for it"
  },
  "eval_2_similar_retrieval": {
    "same_industry_at_5": 0.769,
    "random_baseline": 0.388,
    "lift": 1.98
  },
  "dedupe_rate": 0.0,
  "field_coverage": {
    "website_domain": 0.909,
    "description": 0.913,
    "industry_primary": 1.0,
    "hq_country": 0.975,
    "last_round_type": 0.087,
    "funding_total_usd": 0.078
  },
  "notes": "No real investor-decision or outcome labels exist in free public data, so we validate the system's behavior (thesis responsiveness + comp coherence) rather than realized returns. Swap in real decisions to mea

---
# Milestone 8 — A small API so other tools can use this

Everything so far runs inside this notebook. The final step wraps the results in a tiny
**web service (API)** so other programs — a dashboard, a Slack bot, an n8n workflow — can
ask for data on demand instead of opening the notebook.

We start a small **Flask** server **inside the notebook** (in the background) with three
endpoints:

| Endpoint | What you get |
|---|---|
| `GET /top?k=20` | the top *k* companies by fit score |
| `GET /memo/<company_id>` | a full "deal memo": the company's facts, summary, score breakdown, and its comps |
| `GET /similar/<company_id>?k=5` | the most similar companies (comps) |

Because it runs in the background, the notebook keeps working and we can call the API from
the very next cell to prove it works. (If you'd rather run it as a standalone always-on
service, you can copy this cell's code into a `app.py` file and run `python app.py`.)

In [63]:
import threading
import math
from flask import Flask, jsonify, request

# Make sure every company we serve has the columns the API returns.
serve = companies.copy()
if "summary" not in serve.columns:
    serve["summary"] = None
serve_by_id = {r["company_id"]: r for _, r in serve.iterrows()}


def clean_json(obj):
    """Turn NaN/numpy values into clean JSON (NaN -> null)."""
    if isinstance(obj, float):
        return None if math.isnan(obj) else obj
    if isinstance(obj, dict):
        return {k: clean_json(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [clean_json(v) for v in obj]
    if hasattr(obj, "item"):  # numpy scalar
        return clean_json(obj.item())
    return obj


def company_public(row):
    """The fields we expose for a company."""
    return {
        "company_id": row["company_id"],
        "name": row["name"],
        "website_domain": row.get("website_domain"),
        "score": row.get("score"),
        "stage": row.get("last_round_type"),
        "hq_country": row.get("hq_country"),
        "source": row.get("source"),
        "summary": row.get("summary"),
    }


def build_memo(row):
    """A deal memo: facts + summary + score breakdown + comps."""
    breakdown = {f: row[f"pts_{f}"] for f in FEATURES}
    comps = find_similar(row["company_id"], k=5).to_dict(orient="records")
    return {
        "company": company_public(row),
        "score_breakdown_points": breakdown,
        "description": row.get("description"),
        "similar_companies": comps,
    }


api = Flask("vc_api")


@api.get("/health")
def health():
    return jsonify(clean_json({"ok": True, "companies": len(serve)}))


@api.get("/top")
def api_top():
    k = max(1, min(int(request.args.get("k", 20)), 200))
    items = [company_public(serve.iloc[i]) for i in range(min(k, len(serve)))]
    return jsonify(clean_json({"k": k, "items": items}))


@api.get("/similar/<company_id>")
def api_similar(company_id):
    if company_id not in serve_by_id:
        return jsonify({"error": "company_id not found"}), 404
    k = max(1, min(int(request.args.get("k", 5)), 20))
    comps = find_similar(company_id, k=k).to_dict(orient="records")
    return jsonify(clean_json({"company_id": company_id, "similar_companies": comps}))


@api.get("/memo/<company_id>")
def api_memo(company_id):
    if company_id not in serve_by_id:
        return jsonify({"error": "company_id not found"}), 404
    return jsonify(clean_json(build_memo(serve_by_id[company_id])))


# Start the server once, in the background, so the notebook stays interactive.
API_PORT = 8000
if not globals().get("_api_started"):
    threading.Thread(
        target=lambda: api.run(port=API_PORT, debug=False, use_reloader=False),
        daemon=True,
    ).start()
    _api_started = True
    time.sleep(1.5)
    print(f"API running in the background at http://127.0.0.1:{API_PORT}")
else:
    print("API already running.")

API already running.


### 8a. Call the API to prove it works

Now we act like an outside program and call our own API.

In [64]:
import requests as _rq

base = f"http://127.0.0.1:{API_PORT}"

print("GET /top?k=5")
print(json.dumps(_rq.get(f"{base}/top", params={"k": 5}).json(), indent=2)[:900])

first_id = serve.iloc[0]["company_id"]
print(f"\nGET /memo/{first_id}")
memo = _rq.get(f"{base}/memo/{first_id}").json()
print(json.dumps(memo, indent=2)[:1200])

GET /top?k=5
{
  "items": [
    {
      "company_id": "c_e17b1b5154ed",
      "hq_country": "US",
      "name": "InsForge",
      "score": 58.7,
      "source": "yc",
      "stage": null,
      "summary": "InsForge (B2B) \u2014 InsForge is the agent-native AWS: cloud infrastructure that AI coding agents can understand, operate, and recover from end to end. Momentum signals: 12521 GitHub stars, a top Hacker News post at 62 points. It scores 59/100, driven mainly by thesis match and traction.",
      "website_domain": "insforge.dev"
    },
    {
      "company_id": "c_0bf31361af82",
      "hq_country": "US",
      "name": "Interfaze",
      "score": 55.82,
      "source": "yc",
      "stage": null,
      "summary": "Interfaze (B2B) \u2014 Interfaze is an AI model built on a new architecture that merges specialized DNN/CNN models with transformers for tasks that require deterministic output and high con

GET /memo/c_e17b1b5154ed
{
  "company": {
    "company_id": "c_e17b1b5154ed",
    "hq

### How to call it from a terminal (outside the notebook)

While this notebook (and its server cell) is running, you can hit the same API from a
terminal:
```bash
curl "http://127.0.0.1:8000/top?k=10"
curl "http://127.0.0.1:8000/memo/<company_id>"
curl "http://127.0.0.1:8000/similar/<company_id>?k=5"
```

---
## ✅ Milestones 6, 7 & 8 complete — the pipeline is end-to-end

- **Milestone 6** wrote short, readable **summaries** for the top companies
  (`output/companies_summaries.csv`) — template-based and free, with an optional local-AI
  (Ollama) upgrade.
- **Milestone 7** **evaluated** the system in two honest ways (`output/eval_report.json`):
  the score correctly responds when the thesis changes, and the "similar companies" genuinely
  resemble each other — while being upfront that free data has no real-outcome answer key.
- **Milestone 8** exposed everything through a small **API** (`/top`, `/memo`, `/similar`)
  running inside the notebook.

**The whole journey, entirely on free data:** discover real companies (SEC + YC) → clean &
de-duplicate → enrich with momentum signals (GitHub / HN / jobs) → score & explain → find
comps → summarize → evaluate → serve. No PitchBook, no Harmonic, no paid APIs.

**Ideas for later:** widen the SEC window and enrich more companies for richer traction;
look up websites for SEC companies to unify the two sources; connect the API to Slack/Notion
or an n8n workflow (like the original project); and, when real investor decisions become
available, swap the proxy evaluation for the real thing.

---

# Appendix — Output Files & Column Dictionary

This appendix explains every file the notebook writes to the `output/` folder and what each column means, in plain English.

## The files, and how they build on each other

The pipeline saves the company table at four stages, each one adding more columns to the one before it:

| File | What it is | Rows |
|---|---|---|
| `companies_raw.csv` | Everything collected from SEC + Y Combinator, stacked together (may contain duplicates) | one row per collected record |
| `companies_clean.csv` | Same data after tidying and merging duplicate companies | one row per unique company |
| `companies_enriched.csv` | The clean list **plus** free momentum signals (GitHub, Hacker News, jobs) | one row per unique company |
| `companies_ranked.csv` | The enriched list **plus** the fit score and its full breakdown, sorted best-first | one row per unique company |
| `companies_summaries.csv` | Short written memos for the **top** companies only | one row per top company |
| `similar_example.json` | A worked example of the "find similar companies" feature | 1 example |
| `eval_report.json` | The quality-check results and data-coverage stats | 1 report |

**Where to look for the key things:** the **score and its math** live in `companies_ranked.csv`; the **written memos** live in `companies_summaries.csv`.

## Column dictionary — the company tables

### Identity & description
| Column | Plain-English meaning |
|---|---|
| `company_id` | Unique ID we assign to each company (e.g. `c_e17b1b5154ed`). Used to link across files. |
| `name` | Company name. |
| `website_domain` | Clean web domain (e.g. `stripe.com`). |
| `description` | Short description of what the company does. |
| `industry_primary` | Main industry bucket (e.g. B2B, Fintech, Healthcare). |
| `verticals` | More specific sub-categories / tags. |
| `year_founded` | Year the company was founded (when known). |

### Location
| Column | Plain-English meaning |
|---|---|
| `hq_city` | Headquarters city. |
| `hq_state` | Headquarters state/region. |
| `hq_country` | Headquarters country. |

### Funding (mostly from SEC filings)
| Column | Plain-English meaning |
|---|---|
| `funding_total_usd` | Best estimate of total money raised, in US dollars. |
| `last_round_type` | Inferred stage of the most recent raise (e.g. Pre-Seed, Seed, Early). |
| `last_round_size_usd` | Size of the most recent raise, in US dollars. |
| `last_round_date` | Date of the most recent filing/raise. |
| `filing_count` | How many SEC filings we found for this company. |
| `source` | Where the record came from: `edgar`, `yc`, or `edgar+yc` (found in both). |

### Momentum signals (added in the Enrich step — all free public data)
| Column | Plain-English meaning |
|---|---|
| `github_stars` | Stars on the company's GitHub project (developer interest). |
| `github_activity_90d` | Code activity in the last 90 days (is the project alive?). |
| `hn_mentions` | How many times mentioned on Hacker News. |
| `hn_points` | Total upvotes those Hacker News posts received (buzz). |
| `open_job_postings` | Number of open job listings found (are they growing?). |

### The fit score and its breakdown (added in the Score & Rank step)

First, six **feature values** — each is a normalized 0-to-1 measure of one quality:

| Column | Plain-English meaning |
|---|---|
| `semantic_sim` | How closely the company's description matches your written thesis (0 = unrelated, 1 = perfect match). |
| `stage_fit` | How well its funding stage matches the stage you want. |
| `geo_fit` | How well its location matches where you want to invest. |
| `traction` | Overall momentum, combining the GitHub / Hacker News / jobs signals. |
| `investor_signal` | Strength of investor-related signals (e.g. recent raising activity). |
| `hygiene` | Data completeness/quality for that company (how much we actually know). |

Then the **final score** and how each feature contributed to it:

| Column | Plain-English meaning |
|---|---|
| `score` | The final fit score (higher = better match to your thesis). This is what the ranking sorts on. |
| `pts_semantic_sim` | Points the thesis-match added to the score (feature value × its weight). |
| `pts_stage_fit` | Points the stage-fit added to the score. |
| `pts_geo_fit` | Points the location-fit added to the score. |
| `pts_traction` | Points the momentum signals added to the score. |
| `pts_investor_signal` | Points the investor signal added to the score. |
| `pts_hygiene` | Points the data-quality measure added to the score. |

> The six `pts_*` columns always add up to the `score`, so you can see exactly *why* a company ranked where it did. (These are perfect for a stacked-bar "score breakdown" chart.)

### `companies_summaries.csv` (top companies only)
| Column | Plain-English meaning |
|---|---|
| `company_id`, `name`, `website_domain`, `score` | Same meaning as above. |
| `summary` | A short auto-written memo describing the company and why it fits. |

## `similar_example.json`
A worked example of the "find comparable companies" feature.
- `query_company` — the company we searched from: `company_id`, `name`, `score`.
- `similar_companies` — a list of the closest matches, each with `name`, `industry_primary`, `website_domain`, `score`, and `similarity` (0-to-1 closeness; higher = more similar).

## `eval_report.json`
The pipeline's self-check.
- `thesis` — the investment thesis used for this run.
- `companies_scored` — how many companies were scored.
- `eval_1_thesis_responsiveness` — proof the score reacts to the thesis: the share of a contrast industry (e.g. Healthcare) in the top 30 under your thesis vs. under a thesis that *asks* for that industry. A big jump = the score is really listening to the thesis.
- `eval_2_similar_retrieval` — proof the "similar companies" are coherent: `same_industry_at_5` (how often a company's top-5 comps share its industry) vs. `random_baseline`, and the `lift` (how many times better than random).
- `dedupe_rate` — share of companies that were merged from duplicate records.
- `field_coverage` — for key columns, the fraction of companies that actually have a value (e.g. `description` 0.91 means 91% have a description).
- `notes` — a caveat explaining that no real investor-outcome labels exist in free data, so we validate behavior instead of predicting winners.
